# AIMLCZG546 - Software Engineering for Machine Learning
## Assignment II - AI Resume Screening System
### Group 179

This notebook demonstrates Assignment I continuity, an explicit research
prototype, production refactoring, measured quality, API behavior, and automated
verification.

## Group details

| BITS ID | Name | Contribution | Percentage |
|---|---|---|---:|
| 2025AB05113 | Prashant | ML implementation and API, Architecture and code quality |  |
| 2025AA05032 | Prathap Wagle | ML implementation and API, Architecture and code quality |  |
| 2024AC05999 | Prasanna R T | Test design, execution and review |  |
| 2024AC05914 | Pranav Mehrotra | QA evidence and final review |  |

## 1. Continuity from Assignment I

Assignment II preserves the same resume-screening domain, labelled role data,
TF-IDF/logistic-regression pipeline, explanation rules, upload workflow, human
oversight, and audit history. The monolithic Assignment I implementation is
separated into `ml/`, `services/`, `app/`, tests, and a Streamlit consumer.

## 2. Research code: inline preprocessing prototype

This deliberately compact cell represents notebook-oriented research code. It
has no reusable module boundary, structured logging, type/error contract, size
limit, or test isolation.

In [ ]:
import re

def prototype_clean_text(text):
    return re.sub(r"\\s+", " ", re.sub(r"[^a-z0-9+#. ]", " ", text.lower())).strip()

prototype_clean_text("  Python   TESTING!! ")

### Production version of the same component

The production `clean_text` function is importable, deterministic, logged, typed,
length-limited, and rejects empty, non-string, and semantically empty input. Its
behavior is covered by unit and API-integration tests.

In [ ]:
from ml.preprocessing import clean_text

clean_text("  Python   TESTING!! ")

## 3. Research exploration and baseline comparison

The research path first checks class balance and text-length distribution, then
compares two lightweight text-classification baselines on the same fixed,
stratified holdout. This makes the production choice traceable rather than
presenting a single unexplained model.

In [ ]:
from sklearn.model_selection import train_test_split

from ml.data import build_training_data

frame = build_training_data()
eda = {
    "class_counts": frame["job_role"].value_counts().sort_index().to_dict(),
    "text_length_summary": frame["resume_text"].str.len().describe().round(2).to_dict(),
}
eda

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

train_text, valid_text, train_role, valid_role = train_test_split(
    frame["resume_text"], frame["job_role"], test_size=0.5, random_state=42,
    stratify=frame["job_role"],
)
baseline_results = {}
for name, classifier in {
    "MultinomialNB": MultinomialNB(),
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=42),
}.items():
    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
        ("model", classifier),
    ])
    pipeline.fit(train_text, train_role)
    predicted = pipeline.predict(valid_text)
    baseline_results[name] = {
        "accuracy": accuracy_score(valid_role, predicted),
        "weighted_f1": f1_score(valid_role, predicted, average="weighted"),
    }
baseline_results

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

# The final loop iteration is the selected logistic-regression baseline.
classification_table = pd.DataFrame(
    classification_report(valid_role, predicted, output_dict=True)
).transpose().round(3)
ConfusionMatrixDisplay.from_predictions(
    valid_role,
    predicted,
    xticks_rotation=35,
    colorbar=False,
)
plt.title("Logistic-regression validation confusion matrix")
plt.tight_layout()
plt.show()
classification_table

Logistic regression is retained for production because it provides
stable multiclass probabilities, deterministic training, and interpretable
linear behavior. The small synthetic dataset is workflow evidence, not a claim
of real-world generalization.

## 4. Measured model and data quality

In [ ]:
from pathlib import Path

from logging_config import configure_logging
from ml.data import build_training_data, validate_training_data
from ml.trainer import MIN_ACCURACY, MIN_WEIGHTED_F1, train_model

configure_logging()
frame = build_training_data()
data_quality = validate_training_data(frame)
metrics = train_model(
    frame,
    Path("models/resume_classifier.joblib"),
    Path("reports/metrics_snapshot.json"),
)
data_quality, metrics, {"accuracy_gate": MIN_ACCURACY, "f1_gate": MIN_WEIGHTED_F1}

### Production handoff

`train_model` validates data, evaluates a fixed holdout, enforces accuracy/F1
release gates, retrains on all accepted data, and persists both the model and a
machine-readable metrics snapshot. The runtime reads these versioned artifacts;
it does not train on requests.

## 5. Production inference and REST API contract

In [ ]:
import warnings

warnings.filterwarnings(
    "ignore",
    message=r"Using `httpx` with `starlette.testclient` is deprecated.*",
)

from fastapi.testclient import TestClient

from app.api import app

client = TestClient(app)
health = client.get("/health")
prediction = client.post(
    "/v1/predictions",
    json={
        "candidate_name": "Demo Candidate",
        "resume_text": (
            "QA engineer with Selenium Playwright API testing "
            "and regression automation."
        ),
    },
)
metrics_response = client.get("/metrics")
api_demo = {
    "health": (health.status_code, health.json()),
    "prediction": (prediction.status_code, prediction.json()),
    "metrics": metrics_response.json(),
}
api_demo

The API loads the model once during application startup. `/predict`
and `/v1/predictions` are equivalent contracts, while `/metrics` exposes the
persisted release snapshot. Streamlit and FastAPI share the same application
orchestrator for validation, scoring, and privacy-preserving audit writes.

## 6. Assignment I upload and audit continuity

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

from ml.ingestion import extract_resume_text
from services.audit import AuditRepository

resume_text = extract_resume_text(
    "candidate.txt",
    b"Python machine learning statistics nlp and model evaluation experience.",
)
with TemporaryDirectory() as directory:
    repository = AuditRepository(Path(directory) / "audit.csv")
    repository.save(
        "Demo Candidate",
        "upload",
        resume_text,
        prediction.json(),
    )
    audit_preview = repository.list_results()
audit_preview

## 7. Automated tests and code-quality gates

In [ ]:
import subprocess
import sys

commands = [
    [sys.executable, "-m", "pytest", "-q"],
    [sys.executable, "-m", "black", "--check", "."],
    [sys.executable, "-m", "isort", "--check-only", "."],
    [sys.executable, "-m", "flake8", "."],
]
for command in commands:
    result = subprocess.run(command, capture_output=True, text=True)
    print("$", " ".join(command))
    print(result.stdout or result.stderr or "PASS")
    assert result.returncode == 0

## 8. Production experimentation and security

**Shadow deployment:** mirror production requests to a candidate model without
showing its output to recruiters. Compare latency, error rate, class distribution,
confidence, disagreement, and manually reviewed correctness. Promote only after
predefined gates; then use a small canary with rollback thresholds.

**Security:** enforce schema, type, semantic-content, upload-size, and request-size
limits; never log or persist raw resume text; require TLS, authentication, rate
limiting, malware scanning, isolated file parsing, and least-privilege access to
data and model artifacts. Monitor keyword stuffing, confidence drift, and input
text-length drift; retain a human decision-maker for every recommendation.